In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# TODO: TwoHot over rewards with reward and continue models
# TODO: Implement losses

In [3]:
from collections import defaultdict

import gymnasium as gym
import torch

import fishyrl as frl

In [4]:
# Create environment
envs = gym.vector.AsyncVectorEnv([
    lambda: gym.make('CartPole-v1'),
    lambda: gym.make('CartPole-v1'),
    lambda: gym.make('CartPole-v1'),
    lambda: gym.make('CartPole-v1'),
])

In [5]:
# Initialize model
OBS_DIM = 4
MODEL_ACTIONS = [frl.actions.DiscreteAction(2)]
ACTION_DIM = sum([action.output_dim for action in MODEL_ACTIONS])

EMBEDDED_OBS_DIM = 4

STOCHASTIC_DIM = 6
DETERMINISTIC_DIM = 7
CATEGORICAL_BINS = 11
REWARD_BINS = 13

# Seed
torch.random.manual_seed(42)

# Encoder-decoder models
encoder_model = frl.models.MLPEncoder(OBS_DIM, EMBEDDED_OBS_DIM)
decoder_model = frl.models.MLPDecoder(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, OBS_DIM)

# RSSM models
recurrent_model = frl.models.RecurrentModel(STOCHASTIC_DIM * CATEGORICAL_BINS + ACTION_DIM, DETERMINISTIC_DIM)
representation_model = frl.models.MLP(EMBEDDED_OBS_DIM + DETERMINISTIC_DIM, STOCHASTIC_DIM * CATEGORICAL_BINS)
transition_model = frl.models.MLP(DETERMINISTIC_DIM, STOCHASTIC_DIM * CATEGORICAL_BINS)
rssm_model = frl.models.RSSM(recurrent_model, representation_model, transition_model, CATEGORICAL_BINS)

# Reward and continue models
reward_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, REWARD_BINS, [512])  # Double-check num layers
continue_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, 1, [512])  # Double-check num layers

# Actor and critic models
actor_model = frl.models.Actor(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, MODEL_ACTIONS)
critic_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, REWARD_BINS, [512])

In [ ]:
# Initialize variables
actions = posteriors = hidden_states = initializations = None
rewards, terminations, truncations = 0, None, None
pred_rewards = pred_continues = 0
buffer = defaultdict(list)

# Loop for specified number of iterations
obs, info = envs.reset(seed=42)
for _ in range(10):  # TODO: Test until termination
    # Embed observation
    embedded_obs = encoder_model(torch.tensor(obs))

    # Compute hidden states
    ret = rssm_model(
        actions,
        posteriors,
        hidden_states,
        embedded_obs,
        initializations,
        batch_dim=envs.num_envs,
    )
    hidden_states = ret['hidden_state']
    priors = ret['prior']
    prior_logits = ret['prior_logits']
    posteriors = ret['posterior']
    posterior_logits = ret['posterior_logits']

    # Decode observation
    pred_obs = decoder_model(torch.cat((posteriors, hidden_states), dim=-1))

    # Predict reward and continue
    # TODO: Double-check, but compute reward for the previous action
    pred_rewards = reward_model(torch.cat((posteriors, hidden_states), dim=-1))
    pred_continues = continue_model(torch.cat((posteriors, hidden_states), dim=-1))

    # Compute actions
    actions, action_distributions = actor_model(torch.cat((posteriors, hidden_states), dim=-1))
    frl.actions.simplify_actions(actions, MODEL_ACTIONS).flatten()

    # Record environment-related information to buffer
    buffer['obs'].append(obs)
    buffer['rewards'].append(rewards)
    buffer['terminations'].append(terminations)
    buffer['truncations'].append(truncations)
    buffer['pred_rewards'].append(pred_rewards)
    buffer['pred_continues'].append(pred_continues)

    # Record model-related information to buffer
    buffer['hidden_states'].append(hidden_states)
    buffer['priors'].append(priors)
    buffer['prior_logits'].append(prior_logits)
    buffer['posteriors'].append(posteriors)
    buffer['posterior_logits'].append(posterior_logits)
    buffer['pred_obs'].append(pred_obs)
    buffer['actions'].append(actions)
    buffer['action_distributions'].append(action_distributions)

    # Step environment
    obs, rewards, terminations, truncations, infos = envs.step(
        frl.actions.simplify_actions(actions, MODEL_ACTIONS).flatten().numpy())

In [7]:
envs.close()